# Phase 6 Modeling Consolidated Notebook

This notebook orchestrates **Phase 6 transformer modeling** by calling Python scripts in `scripts/`.

Scope for the current final run:
- Keep **events-only** as the active production path.
- Keep tracking/GNN code available but parked (not deleted).
- Execute final events-only training and exports.
- Restore dropped dual-actor rows via persisted sidecars.
- Write run-scoped outputs to:
  - `Results/Transformer_xT/<run_label>/...`
  - `Models/Transformer_xT/<run_label>/...`

## 1. Set Up Environment and Imports

Install/verify dependencies and set deterministic behavior for reproducible smoke checks.

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import random
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

print(f"Python executable: {sys.executable}")
print(f"UTC now: {datetime.now(timezone.utc).replace(microsecond=0).isoformat()}")
print(f"Torch available: {torch is not None}")

Python executable: x:\Python\.venv\Scripts\python.exe
UTC now: 2026-03-25T20:24:32+00:00
Torch available: True


## 2. Define Configuration and Paths

Configure run label, script paths, and runtime flags so stages are easy to toggle.

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "Data").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory")


def list_existing_runs(results_root: Path) -> list[str]:
    if not results_root.exists():
        return []
    runs = [p for p in results_root.iterdir() if p.is_dir()]
    runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return [p.name for p in runs]


def list_pipeline_run_dirs(runs_root: Path) -> list[Path]:
    if not runs_root.exists():
        return []
    runs = [p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith("run_")]
    runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return runs


BASE_DIR = find_repo_root(Path.cwd())
SCRIPTS_DIR = BASE_DIR / "scripts"
TRAIN_SCRIPT = SCRIPTS_DIR / "train_phase6_transformer_entry.py"
POSTPROCESS_SCRIPT = SCRIPTS_DIR / "postprocess_phase6_outputs.py"
PENALTY_MACRO_SCRIPT = SCRIPTS_DIR / "estimate_penalty_macro_values.py"
THREAT_EXPORT_SCRIPT = "scripts/data_prep/phase3_gnn_embeddings.py"

PIPELINE_RUNS_ROOT = BASE_DIR / "Data" / "Pipeline Runs"
PIPELINE_RUN_DIRS = list_pipeline_run_dirs(PIPELINE_RUNS_ROOT)
if not PIPELINE_RUN_DIRS:
    raise FileNotFoundError(f"No run_* folders found under {PIPELINE_RUNS_ROOT}")

LATEST_PIPELINE_RUN_DIR = PIPELINE_RUN_DIRS[0]
PIPELINE_PHASE2_DIR = LATEST_PIPELINE_RUN_DIR / "phase2"
PIPELINE_PHASE3_DIR = LATEST_PIPELINE_RUN_DIR / "phase3"

PHASE6_RESULTS_ROOT = BASE_DIR / "Results" / "Transformer_xT"
PHASE6_MODELS_ROOT = BASE_DIR / "Models" / "Transformer_xT"

existing_phase6_runs = list_existing_runs(PHASE6_RESULTS_ROOT)
LATEST_PHASE6_RUN = existing_phase6_runs[0] if existing_phase6_runs else None

# Default to resume-friendly behavior. Set True explicitly when forcing a clean benchmark run.
FORCE_NEW_RUN = True
REUSE_RUN_LABEL = "latest"

if FORCE_NEW_RUN:
    RUN_LABEL = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
elif REUSE_RUN_LABEL == "latest":
    RUN_LABEL = LATEST_PHASE6_RUN or datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
else:
    RUN_LABEL = str(REUSE_RUN_LABEL)

RUN_RESULTS_DIR = PHASE6_RESULTS_ROOT / RUN_LABEL
RUN_MODELS_DIR = PHASE6_MODELS_ROOT / RUN_LABEL
RUN_LOGS_DIR = RUN_RESULTS_DIR / "logs"
RUN_METRICS_DIR = RUN_RESULTS_DIR / "metrics"
RUN_INSPECTIONS_DIR = RUN_RESULTS_DIR / "inspections"

for p in [RUN_RESULTS_DIR, RUN_MODELS_DIR, RUN_LOGS_DIR, RUN_METRICS_DIR, RUN_INSPECTIONS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PENALTY_EVENTS_PATH = PIPELINE_PHASE3_DIR / "tensor_ready_dataset.parquet"
PENALTY_WINDOW_SEC = 120.0
PENALTY_OFFSET_EVENT_GAP = 4
REQUIRE_PENALTY_MACRO_TABLE = True
PENALTY_MACRO_REFRESH_POLICY = "if_missing"

BASE_TRAIN_CONFIG = {
    "num_epochs": 100,
    "batch_size": 64,
    "eval_batch_size": 128,
    "num_workers": 4,
    "prefetch_factor": 2,
    "pin_memory": False,
    "persistent_workers": True,
    "debug_validate_items": False,
    "lr": 8e-5,
    "weight_decay": 1e-4,
    "dropout": 0.1,
    "loss_mode": "ce",
    "label_smoothing": 0.005,
    "d_model": 256,
    "n_heads": 8,
    "n_layers": 4,
    "max_seq_length": 128,
    "window_stride": 64,
    "tensorboard_dir": str(BASE_DIR / "TensorBoard" / "Transformer_xT_Training"),
    "seed": 42,
    "resume": True,
    "sample_rows": 0,
    "early_stopping_patience": 10,
    "early_stop_mode": "skill_or_aucpr",
    "early_stop_warmup": 10,
    "aucpr_patience_delta": 1e-4,
    "skill_tolerance": 0.01,
    "focal_gamma": 1.5,
    "loss_balance_power": 0.35,
    "loss_max_ratio": 3.0,
    "rare_class_prob_mass_floor": 0.025,
    "no_goal_prob_ceiling": 0.985,
    "collapse_patience": 4,
    "threat_vector_filename": "threat_vectors.npy",
    "threat_vector_dim": 36,
    "threat_projection_dim": 16,
}

TRACKING_OVERRIDES = {
    "weight_decay": 3e-4,
    "dropout": 0.15,
    "label_smoothing": 0.005,
    "d_model": 256,
    "n_layers": 4,
    "loss_mode": "ce",
}


def resolve_train_config(variant: str) -> dict:
    if variant == "with_tracking":
        return {**BASE_TRAIN_CONFIG, **TRACKING_OVERRIDES}
    return dict(BASE_TRAIN_CONFIG)


RUN_VARIANTS = {
    "events_only": True,
    "with_tracking": False,
}

THREAT_VECTOR_PATH = PIPELINE_PHASE3_DIR / BASE_TRAIN_CONFIG["threat_vector_filename"]

print(f"Base dir: {BASE_DIR}")
print(f"Train script: {TRAIN_SCRIPT}")
print(f"Post-process script: {POSTPROCESS_SCRIPT}")
print(f"Penalty macro script: {PENALTY_MACRO_SCRIPT}")
print(f"Threat export script: {THREAT_EXPORT_SCRIPT}")
print(f"Latest pipeline run: {LATEST_PIPELINE_RUN_DIR.name}")
print(f"Pipeline phase3 dir: {PIPELINE_PHASE3_DIR}")
print(f"Pipeline phase2 dir: {PIPELINE_PHASE2_DIR}")
print(f"Penalty events path: {PENALTY_EVENTS_PATH}")
print(f"Penalty window sec: {PENALTY_WINDOW_SEC}")
print(f"Penalty offset event gap: {PENALTY_OFFSET_EVENT_GAP}")
print(f"Require penalty macro table: {REQUIRE_PENALTY_MACRO_TABLE}")
print(f"Penalty macro refresh policy: {PENALTY_MACRO_REFRESH_POLICY}")
print(f"Threat vectors path: {THREAT_VECTOR_PATH}")
print(f"Run label: {RUN_LABEL}")
print(f"Results dir: {RUN_RESULTS_DIR}")
print(f"Models dir: {RUN_MODELS_DIR}")
print(f"TensorBoard dir: {BASE_TRAIN_CONFIG['tensorboard_dir']}")
print(f"Latest prior run: {LATEST_PHASE6_RUN}")
print("Base training config:")
print(json.dumps(BASE_TRAIN_CONFIG, indent=2))
print("Tracking overrides:")
print(json.dumps(TRACKING_OVERRIDES, indent=2))
print("Run variants:")
print(json.dumps(RUN_VARIANTS, indent=2))

Base dir: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO
Train script: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\scripts\train_phase6_transformer_entry.py
Post-process script: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\scripts\postprocess_phase6_outputs.py
Penalty macro script: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\scripts\estimate_penalty_macro_values.py
Threat export script: scripts/data_prep/phase3_gnn_embeddings.py
Latest pipeline run: run_20260324_200932
Pipeline phase3 dir: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Data\Pipeline Runs\run_20260324_200932\phase3
Pipeline phase2 dir: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Data\Pipeline Runs\run_20260324_200932\phase2
Penalty events path: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Data\Pipeline Runs\run_20260324_200932\phase3\tensor_ready_dataset.parquet
Penalty window sec: 120.0
Penalty offset event gap: 4
Require penalty macro t

In [3]:
# Optimized loader/runtime overrides for this environment.
IS_WINDOWS = sys.platform.startswith("win")
CPU_COUNT = int(os.cpu_count() or 4)

if IS_WINDOWS:
    recommended_workers = max(2, min(4, CPU_COUNT // 2 if CPU_COUNT > 2 else 2))
else:
    recommended_workers = max(4, min(8, CPU_COUNT // 2 if CPU_COUNT > 2 else 2))

xpu_available = bool(torch is not None and hasattr(torch, "xpu") and torch.xpu.is_available())
cuda_available = bool(torch is not None and torch.cuda.is_available())
pin_memory_recommended = bool(cuda_available or xpu_available)

BASE_TRAIN_CONFIG.update(
    {
        "num_workers": int(recommended_workers),
        "prefetch_factor": 2,
        "pin_memory": bool(pin_memory_recommended),
        "persistent_workers": bool(recommended_workers > 0),
        "debug_validate_items": False,
    }
)

print("Applied optimized run overrides:")
print(
    json.dumps(
        {
            "is_windows": IS_WINDOWS,
            "cpu_count": CPU_COUNT,
            "xpu_available": xpu_available,
            "cuda_available": cuda_available,
            "num_workers": BASE_TRAIN_CONFIG["num_workers"],
            "prefetch_factor": BASE_TRAIN_CONFIG["prefetch_factor"],
            "pin_memory": BASE_TRAIN_CONFIG["pin_memory"],
            "persistent_workers": BASE_TRAIN_CONFIG["persistent_workers"],
        },
        indent=2,
    )
)

Applied optimized run overrides:
{
  "is_windows": true,
  "cpu_count": 12,
  "xpu_available": true,
  "cuda_available": false,
  "num_workers": 4,
  "prefetch_factor": 2,
  "pin_memory": true,
  "persistent_workers": true
}


In [4]:
print(f"Post-process script: {POSTPROCESS_SCRIPT}")
print(f"Penalty macro script: {PENALTY_MACRO_SCRIPT}")

Post-process script: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\scripts\postprocess_phase6_outputs.py
Penalty macro script: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\scripts\estimate_penalty_macro_values.py


## 3. Create Core Data Structures

Define small typed containers for stage execution records and artifact tracking.

In [5]:
@dataclass
class StageResult:
    name: str
    enabled: bool
    status: str
    command: str | None = None
    summary_path: str | None = None


@dataclass
class RunArtifacts:
    run_label: str
    results_dir: str
    models_dir: str
    logs_dir: str


artifacts = RunArtifacts(
    run_label=RUN_LABEL,
    results_dir=str(RUN_RESULTS_DIR),
    models_dir=str(RUN_MODELS_DIR),
    logs_dir=str(RUN_LOGS_DIR),
)

display(pd.DataFrame([asdict(artifacts)]))

,run_label,results_dir,models_dir,logs_dir
0,run_20260325_202432,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...


## 4. Implement Core Functions

Add subprocess runner, artifact assertions, and summary loader helpers for stage orchestration.

In [6]:
PYTHON = sys.executable


def run_script(script_rel_path: str, extra_args: list[str] | None = None) -> subprocess.CompletedProcess:
    extra_args = extra_args or []
    cmd = [PYTHON, str(BASE_DIR / script_rel_path)] + extra_args
    print("Running:", " ".join(cmd))

    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.Popen(
        cmd,
        cwd=str(BASE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
        bufsize=1,
    )

    output_lines: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = proc.wait()
    completed = subprocess.CompletedProcess(
        cmd,
        return_code,
        stdout="".join(output_lines),
        stderr=None,
    )

    if completed.returncode != 0:
        raise RuntimeError(f"Script failed ({completed.returncode}): {script_rel_path}")

    return completed


def assert_exists(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected artifact: {path}")
    return path


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _sha256_file(path: Path) -> str:
    import hashlib

    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


PENALTY_MACRO_REFRESH_HISTORY: list[dict] = []


def run_penalty_macro_refresh(
    events_path: Path | None = None,
    window_sec: float = PENALTY_WINDOW_SEC,
    offset_event_gap: int = PENALTY_OFFSET_EVENT_GAP,
    require_table: bool = REQUIRE_PENALTY_MACRO_TABLE,
    refresh_policy: str = PENALTY_MACRO_REFRESH_POLICY,
) -> dict:
    script_rel = "scripts/estimate_penalty_macro_values.py"
    resolved_events_path = Path(events_path) if events_path is not None else PENALTY_EVENTS_PATH
    macro_csv = BASE_DIR / "Results" / "sprint_week" / "penalty_macro_values_by_type.csv"
    report_md = BASE_DIR / "Results" / "sprint_week" / "penalty_macro_estimation_report.md"

    policy_norm = str(refresh_policy).strip().lower()
    if policy_norm not in {"always", "if_missing"}:
        raise ValueError(f"Unsupported penalty macro refresh policy: {refresh_policy}")

    should_refresh = (policy_norm == "always") or (not macro_csv.exists())

    if should_refresh:
        args = [
            "--base-dir", str(BASE_DIR),
            "--window-sec", str(float(window_sec)),
            "--offset-event-gap", str(int(offset_event_gap)),
        ]
        if resolved_events_path:
            args += ["--events-path", str(resolved_events_path)]
        run_script(script_rel, args)
    else:
        print(f"Skipping penalty macro refresh (policy={policy_norm}); using existing table: {macro_csv}")

    assert_exists(macro_csv)

    macro_df = pd.read_csv(macro_csv)
    required_cols = {"penalty_type", "sample_count", "poss_goal_120_rate", "def_goal_120_rate", "penalty_macro_value"}
    missing_cols = sorted(required_cols - set(macro_df.columns))
    if missing_cols:
        raise RuntimeError(f"Penalty macro table missing required columns: {missing_cols}")

    has_default_row = bool(macro_df["penalty_type"].astype(str).str.strip().str.lower().eq("__all__").any())
    if require_table and not has_default_row:
        raise RuntimeError("Penalty macro table missing __all__ fallback row")

    summary = {
        "generated_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "events_path": str(resolved_events_path),
        "window_sec": float(window_sec),
        "offset_event_gap": int(offset_event_gap),
        "refresh_policy": policy_norm,
        "refreshed": bool(should_refresh),
        "macro_csv_path": str(macro_csv),
        "report_path": str(report_md),
        "macro_csv_sha256": _sha256_file(macro_csv),
        "row_count": int(len(macro_df)),
        "has_default_row": bool(has_default_row),
        "types": sorted(macro_df["penalty_type"].astype(str).str.strip().str.lower().unique().tolist()),
    }
    PENALTY_MACRO_REFRESH_HISTORY.append(summary)

    display(pd.DataFrame([summary]))
    return summary

## 5. Preflight Checks for Threat-Vector Runs

Validate that required Phase 3 artifacts exist. If threat vectors are missing, auto-export them before training.

In [7]:
assert TRAIN_SCRIPT.exists(), "train_phase6_transformer_entry.py is missing"
assert POSTPROCESS_SCRIPT.exists(), "postprocess_phase6_outputs.py is missing"
assert PENALTY_MACRO_SCRIPT.exists(), "estimate_penalty_macro_values.py is missing"
assert (PIPELINE_PHASE3_DIR / "tensor_ready_dataset.parquet").exists(), "phase3 tensor_ready_dataset.parquet is missing"
assert (PIPELINE_PHASE3_DIR / "text_embeddings.npy").exists(), "phase3 text_embeddings.npy is missing"

if RUN_VARIANTS.get("with_tracking", False):
    if not THREAT_VECTOR_PATH.exists():
        print(
            "Missing threat vectors for with_tracking; exporting now: "
            f"run_label={LATEST_PIPELINE_RUN_DIR.name}"
        )
        run_script(
            THREAT_EXPORT_SCRIPT,
            [
                "--run-label", LATEST_PIPELINE_RUN_DIR.name,
                "--threat-only",
            ],
        )

    assert THREAT_VECTOR_PATH.exists(), (
        f"Missing threat vectors artifact: {THREAT_VECTOR_PATH}. "
        "Automatic export did not produce the expected file."
    )

    threat_preview = np.load(THREAT_VECTOR_PATH, mmap_mode="r")
    expected_dim = int(BASE_TRAIN_CONFIG.get("threat_vector_dim", 36))
    assert threat_preview.ndim == 2, (
        f"Unexpected threat tensor shape: {threat_preview.shape}. Expected [N, {expected_dim}]."
    )
    assert int(threat_preview.shape[1]) == expected_dim, (
        f"Threat feature width mismatch: expected {expected_dim}, got {threat_preview.shape[1]}"
    )

# Validate helper behavior.
_tmp = RUN_LOGS_DIR / "_tmp_assert_exists.txt"
_tmp.write_text("ok", encoding="utf-8")
assert assert_exists(_tmp) == _tmp
_tmp.unlink(missing_ok=True)

print("Preflight checks passed.")
if RUN_VARIANTS.get("with_tracking", False):
    print(f"Threat vectors ready: {THREAT_VECTOR_PATH}")
    print(f"Threat shape: {tuple(int(x) for x in threat_preview.shape)}")
    print(f"Threat projection dim (model): {BASE_TRAIN_CONFIG['threat_projection_dim']}")

macro_csv = BASE_DIR / "Results" / "sprint_week" / "penalty_macro_values_by_type.csv"
if macro_csv.exists():
    print(f"Existing penalty macro table: {macro_csv}")
else:
    print("Penalty macro table not present yet; it will be generated before each training run.")

Preflight checks passed.
Existing penalty macro table: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\sprint_week\penalty_macro_values_by_type.csv


## 6. Build Variant Training Runner

Define a single training runner that launches events_only and with_tracking variants from the standalone script using the current threat-vector pipeline.

In [8]:
_TRAIN_SCALAR_ARG_MAP = [
    ("num_epochs", "--num-epochs"),
    ("batch_size", "--batch-size"),
    ("eval_batch_size", "--eval-batch-size"),
    ("num_workers", "--num-workers"),
    ("prefetch_factor", "--prefetch-factor"),
    ("lr", "--lr"),
    ("d_model", "--d-model"),
    ("n_heads", "--n-heads"),
    ("n_layers", "--n-layers"),
    ("weight_decay", "--weight-decay"),
    ("dropout", "--dropout"),
    ("loss_mode", "--loss-mode"),
    ("label_smoothing", "--label-smoothing"),
    ("early_stopping_patience", "--early-stopping-patience"),
    ("early_stop_mode", "--early-stop-mode"),
    ("early_stop_warmup", "--early-stop-warmup"),
    ("aucpr_patience_delta", "--aucpr-patience-delta"),
    ("skill_tolerance", "--skill-tolerance"),
    ("focal_gamma", "--focal-gamma"),
    ("loss_balance_power", "--loss-balance-power"),
    ("loss_max_ratio", "--loss-max-ratio"),
    ("rare_class_prob_mass_floor", "--rare-class-prob-mass-floor"),
    ("no_goal_prob_ceiling", "--no-goal-prob-ceiling"),
    ("collapse_patience", "--collapse-patience"),
    ("max_seq_length", "--max-seq-length"),
    ("window_stride", "--window-stride"),
    ("tensorboard_dir", "--tensorboard-dir"),
]


def build_phase6_train_cli_args(
    *,
    run_label: str,
    variant: str,
    config: dict | None = None,
    resume_oof: bool = False,
) -> list[str]:
    cfg = dict(config or {})

    args = [
        "--base-dir", str(BASE_DIR),
        "--run-label", str(run_label),
        "--model-variant", str(variant),
        "--seed", str(cfg.get("seed", BASE_TRAIN_CONFIG.get("seed", SEED))),
    ]

    if resume_oof:
        args += ["--resume", "--num-epochs", "0"]

    for cfg_key, arg_flag in _TRAIN_SCALAR_ARG_MAP:
        if resume_oof and cfg_key == "num_epochs":
            continue
        value = cfg.get(cfg_key)
        if value is not None and value != "":
            args += [arg_flag, str(value)]

    pin_memory = cfg.get("pin_memory")
    if pin_memory is True:
        args += ["--pin-memory"]
    elif pin_memory is False:
        args += ["--no-pin-memory"]

    persistent_workers = cfg.get("persistent_workers")
    if persistent_workers is True:
        args += ["--persistent-workers"]
    elif persistent_workers is False:
        args += ["--no-persistent-workers"]

    debug_validate_items = cfg.get("debug_validate_items")
    if debug_validate_items is True:
        args += ["--debug-validate-items"]
    elif debug_validate_items is False:
        args += ["--no-debug-validate-items"]

    if (not resume_oof) and cfg.get("resume"):
        args += ["--resume"]

    sample_rows = int(cfg.get("sample_rows", 0) or 0)
    if sample_rows > 0:
        args += ["--sample-rows", str(sample_rows)]

    return args


def run_training_variant(
    variant: str,
    config_overrides: dict | None = None,
    run_label_override: str | None = None,
) -> dict:
    if variant not in {"events_only", "with_tracking"}:
        raise ValueError("variant must be 'events_only' or 'with_tracking'")

    config = resolve_train_config(variant)
    if config_overrides:
        config.update(config_overrides)

    run_label = str(run_label_override) if run_label_override else RUN_LABEL
    run_results_dir = PHASE6_RESULTS_ROOT / run_label
    run_models_dir = PHASE6_MODELS_ROOT / run_label
    run_logs_dir = run_results_dir / "logs"

    for p in [run_results_dir, run_models_dir, run_logs_dir]:
        p.mkdir(parents=True, exist_ok=True)

    if variant == "with_tracking":
        assert THREAT_VECTOR_PATH.exists(), f"Missing threat vectors: {THREAT_VECTOR_PATH}"

    penalty_macro_refresh = run_penalty_macro_refresh(
        events_path=PENALTY_EVENTS_PATH,
        window_sec=PENALTY_WINDOW_SEC,
        offset_event_gap=PENALTY_OFFSET_EVENT_GAP,
        require_table=REQUIRE_PENALTY_MACRO_TABLE,
        refresh_policy=PENALTY_MACRO_REFRESH_POLICY,
    )

    args = build_phase6_train_cli_args(
        run_label=run_label,
        variant=variant,
        config=config,
        resume_oof=False,
    )

    print(f"Run label: {run_label}")
    print(f"Variant: {variant}")
    print(f"Penalty macro source CSV: {penalty_macro_refresh['macro_csv_path']}")
    print(f"Penalty macro SHA256: {penalty_macro_refresh['macro_csv_sha256']}")
    print(f"Penalty macro refreshed this run: {penalty_macro_refresh['refreshed']}")
    if variant == "with_tracking":
        print(f"Threat vectors: {THREAT_VECTOR_PATH}")
        print(
            f"Threat dims: in={config['threat_vector_dim']} -> proj={config['threat_projection_dim']} (inside model)"
        )

    print("Resolved training config:")
    print(json.dumps(config, indent=2))

    run_script(str(TRAIN_SCRIPT.relative_to(BASE_DIR)).replace("\\", "/"), args)

    metrics_path = run_results_dir / f"metrics_phase6_{variant}.parquet"
    preds_path = run_results_dir / f"oof_phase6_{variant}.parquet"
    manifest_path = run_logs_dir / f"training_manifest_{variant}.json"

    out = {
        "variant": variant,
        "run_label": run_label,
        "results_dir": str(run_results_dir),
        "models_dir": str(run_models_dir),
        "metrics_path": str(assert_exists(metrics_path)) if metrics_path.exists() else None,
        "preds_path": str(assert_exists(preds_path)) if preds_path.exists() else None,
        "manifest_path": str(assert_exists(manifest_path)) if manifest_path.exists() else None,
        "train_config": config,
        "penalty_macro_refresh": penalty_macro_refresh,
    }

    if metrics_path.exists():
        display(pd.read_parquet(metrics_path))

    return out


training_runs: list[dict] = []
print("Training helper ready. Run baseline and micro-spatial cells below.")

Training helper ready. Run baseline and micro-spatial cells below.


## 7. Execution Cell 1: Events-Only Baseline

Run the baseline model with text and event features only (no tracking-derived threat vectors).

In [18]:
if RUN_VARIANTS["events_only"]:
    training_runs.append(run_training_variant("events_only"))
else:
    print("events_only run is disabled in RUN_VARIANTS.")

Skipping penalty macro refresh (policy=if_missing); using existing table: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\sprint_week\penalty_macro_values_by_type.csv


,generated_at_utc,events_path,window_sec,offset_event_gap,refresh_policy,refreshed,macro_csv_path,report_path,macro_csv_sha256,row_count,has_default_row,types
0,2026-03-25T01:31:12+00:00,x:\My Files\Python\Sports Analytics Projects\H...,120.0,4,if_missing,False,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...,e6d5f34dbf912e072ff2eb7e36731e086e88eae4555186...,2,True,"[__all__, other]"


Run label: run_20260325_013112
Variant: events_only
Penalty macro source CSV: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\sprint_week\penalty_macro_values_by_type.csv
Penalty macro SHA256: e6d5f34dbf912e072ff2eb7e36731e086e88eae4555186c840a520f6444d6df3
Penalty macro refreshed this run: False
Resolved training config:
{
  "num_epochs": 100,
  "batch_size": 64,
  "eval_batch_size": 128,
  "num_workers": 4,
  "prefetch_factor": 2,
  "pin_memory": true,
  "persistent_workers": true,
  "debug_validate_items": false,
  "lr": 8e-05,
  "weight_decay": 0.0001,
  "dropout": 0.1,
  "loss_mode": "ce",
  "label_smoothing": 0.005,
  "d_model": 256,
  "n_heads": 8,
  "n_layers": 4,
  "max_seq_length": 128,
  "window_stride": 64,
  "tensorboard_dir": "x:\\My Files\\Python\\Sports Analytics Projects\\Hockey\\HALO\\TensorBoard\\Transformer_xT_Training",
  "seed": 42,
  "resume": true,
  "sample_rows": 0,
  "early_stopping_patience": 10,
  "early_stop_mode": "skill_or_aucpr",
  "ear

I0000 00:00:1774402278.058877   16324 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774402283.261407   16324 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Using device: xpu
Device name: Intel(R) Arc(TM) B580 Graphics
Using device: xpu
Results dir: X:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\phase6_optimus_reim
Max seq length: 128 | Window stride: 64
Min tokens per window: 10 | Optimus shrinkage K: 200
Feature-engineering coordinate scope: variant=events_only gnn_graph_variant=base
Startup data scope: tracking_merge_enabled=False (model_variant=events_only)
Loading d

## 8. Execution Cell 2: Micro-Spatial Threat-Vector Model

Run the with_tracking variant that consumes phase3 threat_vectors.npy and applies the linear threat projection inside the transformer model.

In [19]:
'''if RUN_VARIANTS["with_tracking"]:
    training_runs.append(run_training_variant("with_tracking"))
else:
    print("with_tracking run is disabled in RUN_VARIANTS.")'''

'if RUN_VARIANTS["with_tracking"]:\n    training_runs.append(run_training_variant("with_tracking"))\nelse:\n    print("with_tracking run is disabled in RUN_VARIANTS.")'

## 9. Decoupled Post-Processing Helpers

Define a minimal post-process runner that:
- Targets fixed run label `run_20260325_013112`
- Optionally refreshes `raw_oof_predictions.parquet` from BEST checkpoints
- Runs `scripts/postprocess_phase6_outputs.py` with only supported arguments
- Validates the new post-process artifacts (`player_ledger.parquet`, `goalie_ledger.parquet`, `consolidated_player_goalie_summary.parquet`)

In [28]:
postprocess_runs: list[dict] = []

# Target the exact finalized run for BEST-checkpoint OOF refresh + postprocess.
POSTPROCESS_TARGET_RUN_LABEL: str = "run_20260325_013112"
POSTPROCESS_PIPELINE_RUN_LABEL: str | None = None
POSTPROCESS_SAMPLE_ROWS: int = 0

# Optional static penalties forwarded to postprocess script.
POSTPROCESS_PENALTY_STATIC_FOR: float | None = None
POSTPROCESS_PENALTY_STATIC_AGAINST: float | None = None

# Refresh raw OOF predictions from BEST checkpoints before postprocess.
POSTPROCESS_RERUN_OOF_PREDS: bool = False
POSTPROCESS_OOF_RERUN_CONFIG_OVERRIDES: dict | None = None


def resolve_postprocess_target_run_label(run_label_override: str | None = None) -> dict:
    run_label = str(run_label_override or POSTPROCESS_TARGET_RUN_LABEL).strip()
    if not run_label:
        raise RuntimeError("POSTPROCESS_TARGET_RUN_LABEL must be a non-empty run label.")
    return {
        "run_label": run_label,
        "source": "fixed_target_run_label",
        "matched_training_run": any(
            isinstance(run, dict)
            and run.get("variant") == "events_only"
            and str(run.get("run_label", "")) == run_label
            for run in training_runs
        ),
    }


def _extract_postprocess_warning_signals(summary: dict) -> dict:
    faceoff_audit = summary.get("faceoff_baseline_audit", {}) if isinstance(summary.get("faceoff_baseline_audit"), dict) else {}
    goalie_audit = summary.get("goalie_goal_penalty_audit", {}) if isinstance(summary.get("goalie_goal_penalty_audit"), dict) else {}

    low_sample_faceoff_zones = faceoff_audit.get("low_sample_zone_warnings", [])
    if not isinstance(low_sample_faceoff_zones, list):
        low_sample_faceoff_zones = []

    freeze_fallback_rows = int(goalie_audit.get("freeze_fallback_rows", 0) or 0)
    goal_link_dedup_rows = int(
        (goalie_audit.get("tensor_goal_key_duplicates_dropped", 0) or 0)
        + (goalie_audit.get("source_link_duplicates_dropped", 0) or 0)
        + (goalie_audit.get("pregoal_link_duplicates_dropped", 0) or 0)
    )

    warning_flags = {
        "warning_low_sample_zone_baseline": bool(faceoff_audit.get("warning_low_sample_zone_baseline", False)),
        "warning_freeze_fallback_rows": bool(freeze_fallback_rows > 0),
        "warning_goal_link_dedup_rows": bool(goal_link_dedup_rows > 0),
    }

    return {
        **warning_flags,
        "total_warning_signals": int(sum(1 for v in warning_flags.values() if v)),
        "low_sample_faceoff_zones": sorted(set(str(x) for x in low_sample_faceoff_zones if str(x).strip())),
        "freeze_fallback_rows": freeze_fallback_rows,
        "goal_link_dedup_rows": goal_link_dedup_rows,
    }


def _expected_events_only_checkpoints(run_models_dir: Path) -> tuple[Path, list[Path]]:
    n_splits = int(BASE_TRAIN_CONFIG.get("n_splits", 5))
    candidate_roots = [run_models_dir, run_models_dir / "checkpoints"]

    for root in candidate_roots:
        checkpoints = [root / f"optimus_reim_A_fold{fold_idx}_best.pth" for fold_idx in range(n_splits)]
        if all(p.exists() for p in checkpoints):
            return root, checkpoints

    best_root = max(
        candidate_roots,
        key=lambda root: sum((root / f"optimus_reim_A_fold{fold_idx}_best.pth").exists() for fold_idx in range(n_splits)),
    )
    checkpoints = [best_root / f"optimus_reim_A_fold{fold_idx}_best.pth" for fold_idx in range(n_splits)]
    return best_root, checkpoints


def _build_oof_refresh_cli_args(run_label: str, config_overrides: dict | None = None) -> list[str]:
    return build_phase6_train_cli_args(
        run_label=run_label,
        variant="events_only",
        config=dict(config_overrides or {}),
        resume_oof=True,
    )


def _refresh_events_only_raw_oof_from_best_checkpoints(
    run_label: str,
    config_overrides: dict | None = None,
) -> dict:
    import shutil

    run_results_dir = PHASE6_RESULTS_ROOT / run_label
    run_models_dir = PHASE6_MODELS_ROOT / run_label

    checkpoint_root, best_checkpoints = _expected_events_only_checkpoints(run_models_dir)
    missing_best = [p for p in best_checkpoints if not p.exists()]
    if missing_best:
        missing_preview = [str(p) for p in missing_best[:5]]
        raise FileNotFoundError(
            "Cannot refresh raw OOF from BEST checkpoints. Missing files: "
            f"{missing_preview}" + (" ..." if len(missing_best) > 5 else "")
        )

    n_splits = int(len(best_checkpoints))
    last_checkpoints = [checkpoint_root / f"optimus_reim_A_fold{fold_idx}_last.pth" for fold_idx in range(n_splits)]

    backup_pairs: list[tuple[Path, Path]] = []
    created_last_paths: list[Path] = []

    try:
        for best_path, last_path in zip(best_checkpoints, last_checkpoints):
            if last_path.exists():
                backup_path = last_path.with_name(last_path.name + ".backup_before_oof_refresh")
                if backup_path.exists():
                    backup_path.unlink()
                shutil.copy2(last_path, backup_path)
                backup_pairs.append((last_path, backup_path))
            else:
                created_last_paths.append(last_path)

            shutil.copy2(best_path, last_path)

        args = _build_oof_refresh_cli_args(
            run_label=str(run_label),
            config_overrides=config_overrides,
        )

        print("Refreshing raw OOF via inference-only resume from BEST checkpoints (num_epochs=0)...")
        run_script(str(TRAIN_SCRIPT.relative_to(BASE_DIR)).replace("\\", "/"), args)

        raw_oof_path = assert_exists(run_results_dir / "raw_oof_predictions.parquet")
        metrics_path = run_results_dir / "metrics_phase6_training.parquet"

        result = {
            "mode": "best_checkpoint_inference_only",
            "run_label": str(run_label),
            "checkpoint_root": str(checkpoint_root),
            "best_checkpoints": [str(p) for p in best_checkpoints],
            "last_checkpoints_refreshed": [str(p) for p in last_checkpoints],
            "raw_oof_path": str(raw_oof_path),
            "metrics_path": str(metrics_path) if metrics_path.exists() else None,
        }

        display(pd.DataFrame([result]))
        return result

    finally:
        for last_path, backup_path in backup_pairs:
            if backup_path.exists():
                shutil.copy2(backup_path, last_path)
                backup_path.unlink(missing_ok=True)

        for last_path in created_last_paths:
            last_path.unlink(missing_ok=True)


def run_postprocess_variant(
    run_label_override: str | None = None,
    pipeline_run_label: str | None = None,
    sample_rows: int = 0,
    penalty_static_for: float | None = None,
    penalty_static_against: float | None = None,
    rerun_oof_preds: bool = POSTPROCESS_RERUN_OOF_PREDS,
    oof_rerun_config_overrides: dict | None = None,
) -> dict:
    target_run = resolve_postprocess_target_run_label(run_label_override=run_label_override)
    postprocess_run_label = str(target_run["run_label"])

    run_results_dir = PHASE6_RESULTS_ROOT / postprocess_run_label
    run_models_dir = PHASE6_MODELS_ROOT / postprocess_run_label
    run_logs_dir = run_results_dir / "logs"

    if not run_results_dir.exists():
        raise FileNotFoundError(
            f"Post-process results directory not found for run_label={postprocess_run_label}: {run_results_dir}"
        )
    if not run_models_dir.exists():
        raise FileNotFoundError(
            f"Post-process models directory not found for run_label={postprocess_run_label}: {run_models_dir}"
        )

    run_logs_dir.mkdir(parents=True, exist_ok=True)

    print(f"Post-process run label: {postprocess_run_label}")
    print(f"Post-process run source: {target_run['source']}")

    oof_refresh_run = None
    if rerun_oof_preds:
        oof_refresh_run = _refresh_events_only_raw_oof_from_best_checkpoints(
            run_label=postprocess_run_label,
            config_overrides=oof_rerun_config_overrides,
        )

    checkpoint_root, expected_checkpoints = _expected_events_only_checkpoints(run_models_dir)
    missing_checkpoints = [p for p in expected_checkpoints if not p.exists()]
    if missing_checkpoints:
        missing_preview = [str(p) for p in missing_checkpoints[:5]]
        raise FileNotFoundError(
            "Cannot run events_only postprocess from model weights. Missing checkpoints: "
            f"{missing_preview}" + (" ..." if len(missing_checkpoints) > 5 else "")
        )

    print(f"Using events_only checkpoints from: {checkpoint_root}")

    args = [
        "--base-dir", str(BASE_DIR),
        "--run-label", postprocess_run_label,
    ]
    if pipeline_run_label:
        args += ["--pipeline-run-label", str(pipeline_run_label)]
    if int(sample_rows) > 0:
        args += ["--sample-rows", str(int(sample_rows))]
    if penalty_static_for is not None:
        args += ["--penalty-static-for", str(float(penalty_static_for))]
    if penalty_static_against is not None:
        args += ["--penalty-static-against", str(float(penalty_static_against))]

    raw_oof_path = assert_exists(run_results_dir / "raw_oof_predictions.parquet")
    run_script(str(POSTPROCESS_SCRIPT.relative_to(BASE_DIR)).replace("\\", "/"), args)

    summary_path = run_logs_dir / "phase6_postprocess_summary.json"
    summary = load_json(assert_exists(summary_path))

    adjusted_path = assert_exists(run_results_dir / "oof_phase6_adjusted_predictions.parquet")
    player_ledger_path = assert_exists(run_results_dir / "player_ledger.parquet")
    goalie_path = assert_exists(run_results_dir / "goalie_ledger.parquet")
    consolidated_summary_path = assert_exists(run_results_dir / "consolidated_player_goalie_summary.parquet")

    faceoff_inspection_csv = summary.get("faceoff_baselines_inspection_csv")
    export_paths = {
        str(adjusted_path),
        str(player_ledger_path),
        str(goalie_path),
        str(consolidated_summary_path),
        str(summary_path),
    }
    if isinstance(faceoff_inspection_csv, str) and faceoff_inspection_csv.strip():
        export_paths.add(faceoff_inspection_csv)

    rows = summary.get("rows", {}) if isinstance(summary.get("rows"), dict) else {}
    warning_signals = _extract_postprocess_warning_signals(summary)

    out = {
        "variant": "events_only",
        "run_label": postprocess_run_label,
        "postprocess_target_source": str(target_run.get("source", "unknown")),
        "postprocess_target_matched_training_run": bool(target_run.get("matched_training_run", False)),
        "summary_path": str(summary_path),
        "results_dir": str(run_results_dir),
        "models_dir": str(run_models_dir),
        "raw_oof_path": str(raw_oof_path),
        "export_paths": sorted(export_paths),
        "rerun_oof_preds": bool(rerun_oof_preds),
        "oof_refresh_mode": str(oof_refresh_run.get("mode")) if isinstance(oof_refresh_run, dict) and oof_refresh_run.get("mode") else None,
        "oof_refresh_checkpoint_root": str(oof_refresh_run.get("checkpoint_root")) if isinstance(oof_refresh_run, dict) and oof_refresh_run.get("checkpoint_root") else None,
        "adjusted_rows": int(rows.get("adjusted", 0)),
        "player_ledger_rows": int(rows.get("player_ledger", 0)),
        "goalie_ledger_rows": int(rows.get("goalie_ledger", 0)),
        "consolidated_rows": int(rows.get("consolidated_player_goalie_summary", 0)),
        **warning_signals,
    }

    if out["total_warning_signals"] > 0:
        print("Notebook warning summary from postprocess audit:")
        print(f"  - low_sample_faceoff_zones: {out['low_sample_faceoff_zones']}")
        print(f"  - freeze_fallback_rows: {out['freeze_fallback_rows']}")
        print(f"  - goal_link_dedup_rows: {out['goal_link_dedup_rows']}")

    display(
        pd.DataFrame(
            [
                {
                    "run_label": out["run_label"],
                    "source": out["postprocess_target_source"],
                    "rerun_oof_preds": out["rerun_oof_preds"],
                    "oof_refresh_mode": out["oof_refresh_mode"],
                    "exports_count": len(out["export_paths"]),
                    "summary_path": out["summary_path"],
                    "adjusted_rows": out["adjusted_rows"],
                    "player_ledger_rows": out["player_ledger_rows"],
                    "goalie_ledger_rows": out["goalie_ledger_rows"],
                    "consolidated_rows": out["consolidated_rows"],
                    "warning_signals": out["total_warning_signals"],
                }
            ]
        )
    )
    return out

## 10. Post-Process Cell 1: Events-Only Reprocess

Refresh raw OOF predictions from BEST checkpoints for `run_20260325_013112`, then execute post-process to produce adjusted predictions, ledgers, and summary outputs.

In [29]:
if RUN_VARIANTS["events_only"]:
    postprocess_target = resolve_postprocess_target_run_label()
    print(
        "Post-process target resolved for Section 10: "
        f"run_label={postprocess_target['run_label']} "
        f"(source={postprocess_target['source']})"
    )

    postprocess_runs.append(
        run_postprocess_variant(
            run_label_override=str(postprocess_target["run_label"]),
            pipeline_run_label=POSTPROCESS_PIPELINE_RUN_LABEL,
            sample_rows=POSTPROCESS_SAMPLE_ROWS,
            penalty_static_for=POSTPROCESS_PENALTY_STATIC_FOR,
            penalty_static_against=POSTPROCESS_PENALTY_STATIC_AGAINST,
            rerun_oof_preds=POSTPROCESS_RERUN_OOF_PREDS,
            oof_rerun_config_overrides=POSTPROCESS_OOF_RERUN_CONFIG_OVERRIDES,
        )
    )
else:
    print("events_only post-process run is disabled in RUN_VARIANTS.")

Post-process target resolved for Section 10: run_label=run_20260325_013112 (source=fixed_target_run_label)
Post-process run label: run_20260325_013112
Post-process run source: fixed_target_run_label
Using events_only checkpoints from: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Models\Transformer_xT\run_20260325_013112\checkpoints
Running: x:\Python\.venv\Scripts\python.exe x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\scripts\postprocess_phase6_outputs.py --base-dir x:\My Files\Python\Sports Analytics Projects\Hockey\HALO --run-label run_20260325_013112
Postprocess complete.
  Player ledger:   X:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260325_013112\player_ledger.parquet
  Goalie ledger:   X:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260325_013112\goalie_ledger.parquet
  Summary:         X:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260325

,run_label,source,rerun_oof_preds,oof_refresh_mode,exports_count,summary_path,adjusted_rows,player_ledger_rows,goalie_ledger_rows,consolidated_rows,warning_signals
0,run_20260325_013112,fixed_target_run_label,False,None,5,x:\My Files\Python\Sports Analytics Projects\H...,1688117,1629407,30471,0,0


## 11. Post-Process Cell 2: With-Tracking Exports

This section is intentionally deprecated for the current events-only workflow and remains as a no-op compatibility placeholder.

In [ ]:
print("with_tracking post-process is deprecated for this workflow and is intentionally skipped.")

## 12. Post-Process Cell 3: Combined Exports

This section is intentionally deprecated for the current events-only workflow and remains as a no-op compatibility placeholder.

In [ ]:
print("combined post-process is deprecated for this workflow and is intentionally skipped.")

## 13. Persist Run Manifest

Save run metadata and key artifacts produced by the split training plus decoupled post-processing cells.

In [ ]:
artifact_candidates = [
    RUN_RESULTS_DIR / "oof_phase6_events_only.parquet",
    RUN_RESULTS_DIR / "metrics_phase6_events_only.parquet",
    RUN_RESULTS_DIR / "oof_phase6_with_tracking.parquet",
    RUN_RESULTS_DIR / "metrics_phase6_with_tracking.parquet",
    RUN_RESULTS_DIR / "oof_phase6_predictions.parquet",
    RUN_RESULTS_DIR / "metrics_phase6_all_models.parquet",
    RUN_RESULTS_DIR / "optimus_reim_event_predictions_events_only.parquet",
    RUN_RESULTS_DIR / "optimus_reim_event_predictions_with_tracking.parquet",
    RUN_RESULTS_DIR / "optimus_reim_event_predictions.parquet",
    RUN_RESULTS_DIR / "phase6_export_summary_events_only.json",
    RUN_RESULTS_DIR / "phase6_export_summary_with_tracking.json",
    RUN_RESULTS_DIR / "phase6_export_summary.json",
    RUN_RESULTS_DIR / "oof_phase6_adjusted_predictions.parquet",
    RUN_RESULTS_DIR / "player_ledger.parquet",
    RUN_RESULTS_DIR / "goalie_ledger.parquet",
    RUN_RESULTS_DIR / "consolidated_player_goalie_summary.parquet",
    RUN_LOGS_DIR / "phase6_postprocess_summary.json",
    RUN_INSPECTIONS_DIR / "faceoff_baselines_inspection.csv",
    RUN_LOGS_DIR / "phase6_smoke_preflight_summary.json",
    BASE_DIR / "Results" / "sprint_week" / "penalty_macro_values_by_type.csv",
    BASE_DIR / "Results" / "sprint_week" / "penalty_macro_estimation_report.md",
]

extra_run_artifacts: list[str] = []
for run in training_runs:
    for key in ("metrics_path", "preds_path", "manifest_path"):
        path_value = run.get(key)
        if path_value:
            extra_run_artifacts.append(str(path_value))

postprocess_runs_safe = postprocess_runs if "postprocess_runs" in globals() else []
for run in postprocess_runs_safe:
    summary_path = run.get("summary_path")
    if summary_path:
        extra_run_artifacts.append(str(summary_path))
    for path_value in run.get("export_paths", []):
        if path_value:
            extra_run_artifacts.append(str(path_value))

available_artifacts = sorted(set([str(p) for p in artifact_candidates if p.exists()] + extra_run_artifacts))

latest_penalty_macro_refresh = PENALTY_MACRO_REFRESH_HISTORY[-1] if PENALTY_MACRO_REFRESH_HISTORY else None

run_manifest = {
    "generated_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "run_label": RUN_LABEL,
    "train_script": str(TRAIN_SCRIPT),
    "postprocess_script": str(POSTPROCESS_SCRIPT),
    "penalty_macro_script": str(PENALTY_MACRO_SCRIPT),
    "threat_export_script": THREAT_EXPORT_SCRIPT,
    "results_dir": str(RUN_RESULTS_DIR),
    "models_dir": str(RUN_MODELS_DIR),
    "base_train_config": BASE_TRAIN_CONFIG,
    "tracking_overrides": TRACKING_OVERRIDES,
    "penalty_macro_config": {
        "events_path": str(PENALTY_EVENTS_PATH),
        "window_sec": float(PENALTY_WINDOW_SEC),
        "offset_event_gap": int(PENALTY_OFFSET_EVENT_GAP),
        "require_table": bool(REQUIRE_PENALTY_MACRO_TABLE),
    },
    "latest_penalty_macro_refresh": latest_penalty_macro_refresh,
    "penalty_macro_refresh_history": PENALTY_MACRO_REFRESH_HISTORY,
    "threat_vectors": {
        "path": str(THREAT_VECTOR_PATH),
        "expected_dim": int(BASE_TRAIN_CONFIG["threat_vector_dim"]),
        "projection_dim": int(BASE_TRAIN_CONFIG["threat_projection_dim"]),
        "exists": bool(THREAT_VECTOR_PATH.exists()),
    },
    "run_variants": RUN_VARIANTS,
    "training_runs": training_runs,
    "postprocess_runs": postprocess_runs_safe,
    "available_artifacts": available_artifacts,
}

manifest_path = RUN_LOGS_DIR / "phase6_modeling_run_manifest.json"
with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print(f"Saved: {manifest_path}")
print("\nAvailable run artifacts:")
for path_str in available_artifacts:
    print("-", path_str)

if training_runs:
    display(pd.DataFrame(training_runs))
else:
    print("No training variants have been executed yet.")

if postprocess_runs_safe:
    display(pd.DataFrame(postprocess_runs_safe))

    warning_digest_rows: list[dict] = []
    for run in postprocess_runs_safe:
        warning_digest_rows.append(
            {
                "variant": str(run.get("variant", "")),
                "run_label": str(run.get("run_label", "")),
                "warning_signals": int(run.get("total_warning_signals", 0) or 0),
                "low_sample_faceoff_zones": ",".join(run.get("low_sample_faceoff_zones", []) or []),
                "freeze_fallback_rows": int(run.get("freeze_fallback_rows", 0) or 0),
                "goal_link_dedup_rows": int(run.get("goal_link_dedup_rows", 0) or 0),
            }
        )

    print("\nPost-process warning digest:")
    display(pd.DataFrame(warning_digest_rows))
else:
    print("No post-process variants have been executed yet.")

Saved: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260324_150347\logs\phase6_modeling_run_manifest.json

Available run artifacts:
- x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260324_150347\logs\phase6_postprocess_summary.json
- x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260324_150347\logs\phase6_postprocess_summary_events_only.json
- x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260324_150347\logs\training_manifest_events_only.json
- x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260324_150347\oof_phase6_adjusted_predictions.parquet
- x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260324_150347\optimus_reim_global_player_credit_ledger_events_only.parquet
- x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\run_20260

,variant,run_label,results_dir,models_dir,metrics_path,preds_path,manifest_path,train_config,penalty_macro_refresh
0,events_only,run_20260324_150347,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...,None,None,x:\My Files\Python\Sports Analytics Projects\H...,"{'num_epochs': 100, 'batch_size': 64, 'eval_ba...",{'generated_at_utc': '2026-03-24T15:04:03+00:0...


,variant,run_label,postprocess_target_source,postprocess_target_matched_training_run,run_label_override_requested,summary_path,summary_path_canonical,results_dir,models_dir,raw_oof_path,...,warning_freeze_fallback_rows,warning_goal_link_dedup_rows,total_warning_signals,low_sample_faceoff_zones,freeze_fallback_rows,save_rows_missing_post_save_context,goal_link_dedup_rows,tensor_goal_key_duplicates_dropped,source_link_duplicates_dropped,pregoal_link_duplicates_dropped
0,events_only,run_20260324_150347,manual_override,False,run_20260324_150347,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...,x:\My Files\Python\Sports Analytics Projects\H...,...,True,False,1,[],24,30,0,0,0,0



Post-process warning digest:


,variant,run_label,warning_signals,low_sample_faceoff_zones,freeze_fallback_rows,goal_link_dedup_rows
0,events_only,run_20260324_150347,1,,24,0
